# Random Forest Model Predictions & Visualizations

This notebook loads the trained Random Forest model and creates comprehensive visualizations of predictions and model performance on air quality alarm data.

**Dataset:** 12,109 sensor readings from 4 locations (March 2026)
**Features:** PM2.5, PM10, Temperature, Humidity, Gas (MQ2), Carbon Monoxide (MQ7)
**Target:** Alarm Status (0 = No Alarm, 1 = Alarm)

## 1. Import Libraries & Load Saved Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_curve, roc_auc_score,
    precision_recall_curve, f1_score, classification_report
)

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

# Load saved model, scaler, and feature names
models_dir = '../models'
dataset_dir = '../dataset'

# Load model
model_path = os.path.join(models_dir, 'random_forest_model.pkl')
with open(model_path, 'rb') as f:
    trained_model = pickle.load(f)
print(f"✓ Model loaded from: {model_path}")

# Load scaler
scaler_path = os.path.join(models_dir, 'scaler.pkl')
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)
print(f"✓ Scaler loaded from: {scaler_path}")

# Feature names
features = ['pm2_5', 'pm10', 'temp', 'humidity', 'gas', 'co']
print(f"✓ Features: {features}")
print(f"✓ All models loaded successfully!")

## 2. Load Test Data & Make Predictions

In [ ]:
# Load combined dataset
dataset_path = os.path.join(dataset_dir, 'combined_data.csv')
df = pd.read_csv(dataset_path)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")
print(f"\nAlarm distribution:")
print(df['alarm_status'].value_counts())
print(f"\nDataset info:")
print(df.info())

# Prepare data for predictions
X = df[features].values
y = df['alarm_status'].values

# Split data (same as training)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale test data
X_test_scaled = scaler.transform(X_test)

# Make predictions
y_pred = trained_model.predict(X_test_scaled)
y_pred_proba = trained_model.predict_proba(X_test_scaled)[:, 1]

print(f"\n✓ Predictions made on {len(y_test)} test samples")
print(f"  - Predicted Alarms: {(y_pred == 1).sum()}")
print(f"  - Predicted No Alarm: {(y_pred == 0).sum()}")
print(f"\n  Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  F1 Score: {f1_score(y_test, y_pred):.4f}")

## 3. Prediction Distribution Visualizations

In [ ]:
# Create figure for prediction distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Model Prediction Distributions & Analysis', fontsize=16, fontweight='bold')

# 1. Prediction Probability Distribution
ax1 = axes[0, 0]
ax1.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.6, label='No Alarm (Actual)', color='green', edgecolor='black')
ax1.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.6, label='Alarm (Actual)', color='red', edgecolor='black')
ax1.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
ax1.set_xlabel('Prediction Probability (Alarm Class)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax1.set_title('Prediction Probability Distribution by Actual Class', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Alarm vs No-Alarm Predictions (Bar Chart)
ax2 = axes[0, 1]
pred_counts = pd.DataFrame({
    'Actual No Alarm': [((y_test == 0) & (y_pred == 0)).sum(), ((y_test == 0) & (y_pred == 1)).sum()],
    'Actual Alarm': [((y_test == 1) & (y_pred == 0)).sum(), ((y_test == 1) & (y_pred == 1)).sum()]
}, index=['Predicted No Alarm', 'Predicted Alarm'])
pred_counts.T.plot(kind='bar', ax=ax2, color=['green', 'red'], alpha=0.7, edgecolor='black')
ax2.set_title('Prediction Counts: Actual vs Predicted', fontsize=12, fontweight='bold')
ax2.set_xlabel('Actual Class', fontsize=11, fontweight='bold')
ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
ax2.legend(title='Prediction', loc='upper right')
ax2.grid(True, alpha=0.3, axis='y')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Confidence Score Distribution
ax3 = axes[1, 0]
confidence_scores = np.maximum(y_pred_proba, 1 - y_pred_proba)
ax3.hist(confidence_scores, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax3.axvline(confidence_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {confidence_scores.mean():.3f}')
ax3.set_xlabel('Confidence Score', fontsize=11, fontweight='bold')
ax3.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax3.set_title('Model Confidence Score Distribution', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Prediction Probability Distribution (Violin Plot)
ax4 = axes[1, 1]
violin_data = [y_pred_proba[y_test == 0], y_pred_proba[y_test == 1]]
parts = ax4.violinplot(violin_data, positions=[1, 2], showmeans=True, showmedians=True)
ax4.set_xticks([1, 2])
ax4.set_xticklabels(['No Alarm', 'Alarm'])
ax4.set_ylabel('Prediction Probability', fontsize=11, fontweight='bold')
ax4.set_title('Prediction Probability Distribution (Violin Plot)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✓ Prediction distribution visualizations completed")

## 4. Feature-Based Prediction Analysis

In [ ]:
# Create feature analysis visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Feature-Based Prediction Analysis - Scatter Plots with Predictions', fontsize=16, fontweight='bold')

# Create test data dataframe for easier plotting
X_test_df = pd.DataFrame(X_test, columns=features)
X_test_df['actual'] = y_test
X_test_df['predicted'] = y_pred
X_test_df['probability'] = y_pred_proba

# 1. PM2.5 vs PM10 colored by actual class
ax1 = axes[0, 0]
scatter1 = ax1.scatter(X_test_df[X_test_df['actual'] == 0]['pm2_5'], 
                       X_test_df[X_test_df['actual'] == 0]['pm10'],
                       c='green', alpha=0.6, s=60, label='Actual: No Alarm', edgecolors='darkgreen')
scatter1 = ax1.scatter(X_test_df[X_test_df['actual'] == 1]['pm2_5'], 
                       X_test_df[X_test_df['actual'] == 1]['pm10'],
                       c='red', alpha=0.6, s=60, label='Actual: Alarm', edgecolors='darkred')
ax1.set_xlabel('PM2.5 (μg/m³)', fontsize=11, fontweight='bold')
ax1.set_ylabel('PM10 (μg/m³)', fontsize=11, fontweight='bold')
ax1.set_title('PM2.5 vs PM10 - Colored by Actual Class', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. PM2.5 vs PM10 colored by prediction
ax2 = axes[0, 1]
ax2.scatter(X_test_df[X_test_df['predicted'] == 0]['pm2_5'], 
           X_test_df[X_test_df['predicted'] == 0]['pm10'],
           c='lightgreen', alpha=0.6, s=60, label='Predicted: No Alarm', edgecolors='darkgreen')
ax2.scatter(X_test_df[X_test_df['predicted'] == 1]['pm2_5'], 
           X_test_df[X_test_df['predicted'] == 1]['pm10'],
           c='lightcoral', alpha=0.6, s=60, label='Predicted: Alarm', edgecolors='darkred')
ax2.set_xlabel('PM2.5 (μg/m³)', fontsize=11, fontweight='bold')
ax2.set_ylabel('PM10 (μg/m³)', fontsize=11, fontweight='bold')
ax2.set_title('PM2.5 vs PM10 - Colored by Prediction', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Temperature vs Humidity colored by prediction confidence
ax3 = axes[0, 2]
scatter3 = ax3.scatter(X_test_df['temp'], X_test_df['humidity'], 
                       c=X_test_df['probability'], cmap='RdYlGn_r', s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
ax3.set_xlabel('Temperature (°C)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Humidity (%)', fontsize=11, fontweight='bold')
ax3.set_title('Temperature vs Humidity - Colored by Alarm Probability', fontsize=12, fontweight='bold')
cbar3 = plt.colorbar(scatter3, ax=ax3)
cbar3.set_label('Alarm Probability', fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Gas vs CO colored by actual class
ax4 = axes[1, 0]
ax4.scatter(X_test_df[X_test_df['actual'] == 0]['gas'], 
           X_test_df[X_test_df['actual'] == 0]['co'],
           c='green', alpha=0.6, s=60, label='Actual: No Alarm', edgecolors='darkgreen')
ax4.scatter(X_test_df[X_test_df['actual'] == 1]['gas'], 
           X_test_df[X_test_df['actual'] == 1]['co'],
           c='red', alpha=0.6, s=60, label='Actual: Alarm', edgecolors='darkred')
ax4.set_xlabel('Gas (ppm)', fontsize=11, fontweight='bold')
ax4.set_ylabel('CO (ppm)', fontsize=11, fontweight='bold')
ax4.set_title('Gas vs CO - Colored by Actual Class', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. PM2.5 vs Temperature
ax5 = axes[1, 1]
scatter5 = ax5.scatter(X_test_df[X_test_df['predicted'] == 0]['pm2_5'], 
                       X_test_df[X_test_df['predicted'] == 0]['temp'],
                       c='lightgreen', alpha=0.6, s=60, label='Predicted: No Alarm', edgecolors='darkgreen')
scatter5 = ax5.scatter(X_test_df[X_test_df['predicted'] == 1]['pm2_5'], 
                       X_test_df[X_test_df['predicted'] == 1]['temp'],
                       c='lightcoral', alpha=0.6, s=60, label='Predicted: Alarm', edgecolors='darkred')
ax5.set_xlabel('PM2.5 (μg/m³)', fontsize=11, fontweight='bold')
ax5.set_ylabel('Temperature (°C)', fontsize=11, fontweight='bold')
ax5.set_title('PM2.5 vs Temperature - Colored by Prediction', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Feature Importance
ax6 = axes[1, 2]
feature_importance = trained_model.feature_importances_
feature_names = features
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance}).sort_values('Importance', ascending=True)
colors_importance = ['red' if x > 0.2 else 'steelblue' for x in importance_df['Importance']]
ax6.barh(importance_df['Feature'], importance_df['Importance'], color=colors_importance, edgecolor='black', alpha=0.7)
ax6.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
ax6.set_title('Random Forest Feature Importance', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("✓ Feature-based prediction analysis completed")

## 5. Model Performance Dashboard

In [ ]:
# Calculate performance metrics
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc, 
                           precision_recall_curve, average_precision_score)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

# Create performance dashboard
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
fig.suptitle('Model Performance Dashboard', fontsize=18, fontweight='bold')

# 1. Confusion Matrix (Raw Counts)
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1, 
            xticklabels=['No Alarm', 'Alarm'], yticklabels=['No Alarm', 'Alarm'],
            annot_kws={'size': 14, 'weight': 'bold'})
ax1.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax1.set_title('Confusion Matrix (Raw Counts)', fontsize=12, fontweight='bold')

# 2. Confusion Matrix (Normalized)
ax2 = fig.add_subplot(gs[0, 1])
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='RdYlGn', cbar=True, ax=ax2,
            xticklabels=['No Alarm', 'Alarm'], yticklabels=['No Alarm', 'Alarm'],
            annot_kws={'size': 12, 'weight': 'bold'})
ax2.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax2.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax2.set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')

# 3. Performance Metrics Table
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
metrics_data = [
    ['Metric', 'Value'],
    ['Accuracy', f'{accuracy:.4f}'],
    ['Precision', f'{precision:.4f}'],
    ['Recall (Sensitivity)', f'{recall:.4f}'],
    ['Specificity', f'{tn / (tn + fp):.4f}'],
    ['F1 Score', f'{f1:.4f}'],
    ['ROC AUC', f'{roc_auc:.4f}'],
    ['Avg Precision', f'{avg_precision:.4f}'],
    ['True Positives', f'{tp}'],
    ['True Negatives', f'{tn}'],
    ['False Positives', f'{fp}'],
    ['False Negatives', f'{fn}']
]
colors = [['#40466e', 'white']] + [['lightgray', 'black']] * (len(metrics_data) - 1)
table = ax3.table(cellText=metrics_data, cellLoc='center', loc='center', 
                 cellColours=colors, bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)
for i, row in enumerate(metrics_data):
    if i == 0:
        for j in range(len(row)):
            table[(i, j)].set_text_props(weight='bold', color='white', size=11)
ax3.set_title('Performance Metrics Summary', fontsize=12, fontweight='bold', pad=20)

# 4. ROC Curve
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax4.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax4.set_xlim([0.0, 1.0])
ax4.set_ylim([0.0, 1.05])
ax4.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax4.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax4.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax4.legend(loc="lower right", fontsize=10)
ax4.grid(True, alpha=0.3)

# 5. Precision-Recall Curve
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(recall_vals, precision_vals, color='darkgreen', lw=2.5, label=f'PR curve (AP = {avg_precision:.4f})')
ax5.axhline(y=recall.sum() / len(y_test), color='navy', linestyle='--', lw=2, label='Random Classifier')
ax5.set_xlabel('Recall', fontsize=11, fontweight='bold')
ax5.set_ylabel('Precision', fontsize=11, fontweight='bold')
ax5.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax5.set_xlim([0.0, 1.0])
ax5.set_ylim([0.0, 1.05])
ax5.legend(loc="upper right", fontsize=10)
ax5.grid(True, alpha=0.3)

# 6. Prediction Distribution with Decision Threshold
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7, label='No Alarm', color='green', edgecolor='black')
ax6.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7, label='Alarm', color='red', edgecolor='black')
ax6.axvline(0.5, color='black', linestyle='--', linewidth=2.5, label='Decision Threshold (0.5)')
ax6.set_xlabel('Prediction Probability', fontsize=11, fontweight='bold')
ax6.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax6.set_title('Prediction Probability Distribution', fontsize=12, fontweight='bold')
ax6.legend(fontsize=10)
ax6.grid(True, alpha=0.3, axis='y')

plt.show()

print("✓ Model performance dashboard completed")
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Alarm', 'Alarm']))

## 6. Real-Time Prediction Scenarios

In [ ]:
# Define real-world sensor scenarios
scenarios = {
    'Normal Conditions': {'pm2_5': 15, 'pm10': 30, 'temp': 25, 'humidity': 50, 'gas': 80, 'co': 2},
    'Light Pollution': {'pm2_5': 50, 'pm10': 80, 'temp': 22, 'humidity': 45, 'gas': 100, 'co': 3},
    'Moderate Pollution': {'pm2_5': 85, 'pm10': 150, 'temp': 28, 'humidity': 65, 'gas': 140, 'co': 5},
    'High Pollution Alert': {'pm2_5': 120, 'pm10': 250, 'temp': 32, 'humidity': 75, 'gas': 180, 'co': 8},
    'Extreme Air Quality': {'pm2_5': 200, 'pm10': 400, 'temp': 35, 'humidity': 85, 'gas': 250, 'co': 15},
    'Cold & Humid': {'pm2_5': 40, 'pm10': 70, 'temp': 5, 'humidity': 80, 'gas': 90, 'co': 4},
    'Hot & Dry': {'pm2_5': 60, 'pm10': 110, 'temp': 38, 'humidity': 20, 'gas': 120, 'co': 6},
    'Industrial Spike': {'pm2_5': 150, 'pm10': 280, 'temp': 24, 'humidity': 55, 'gas': 220, 'co': 12}
}

# Make predictions for each scenario
scenario_predictions = {}
for scenario_name, values in scenarios.items():
    scenario_array = np.array([[values[f] for f in features]])
    scenario_scaled = scaler.transform(scenario_array)
    pred = trained_model.predict(scenario_scaled)[0]
    pred_proba = trained_model.predict_proba(scenario_scaled)[0][1]
    scenario_predictions[scenario_name] = {
        'prediction': 'ALARM' if pred == 1 else 'NO ALARM',
        'probability': pred_proba,
        'confidence': max(pred_proba, 1 - pred_proba)
    }

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Real-Time Scenario Predictions - Emergency Response Examples', fontsize=16, fontweight='bold')

# 1. Scenario Names and Predictions (Bar Chart)
ax1 = axes[0, 0]
scenario_names = list(scenario_predictions.keys())
probabilities = [scenario_predictions[s]['probability'] for s in scenario_names]
colors_scenario = ['red' if p > 0.5 else 'green' for p in probabilities]
bars = ax1.barh(scenario_names, probabilities, color=colors_scenario, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
ax1.set_xlabel('Alarm Probability', fontsize=11, fontweight='bold')
ax1.set_title('Scenario-Based Alarm Predictions', fontsize=12, fontweight='bold')
ax1.set_xlim([0, 1])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='x')
for i, (bar, prob) in enumerate(zip(bars, probabilities)):
    ax1.text(prob + 0.02, i, f'{prob:.3f}', va='center', fontweight='bold', fontsize=9)

# 2. Confidence Score Comparison
ax2 = axes[0, 1]
confidences = [scenario_predictions[s]['confidence'] for s in scenario_names]
colors_conf = ['darkred' if p > 0.7 else 'steelblue' for p in confidences]
bars2 = ax2.barh(scenario_names, confidences, color=colors_conf, alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Model Confidence', fontsize=11, fontweight='bold')
ax2.set_title('Model Confidence per Scenario', fontsize=12, fontweight='bold')
ax2.set_xlim([0, 1])
ax2.grid(True, alpha=0.3, axis='x')
for i, (bar, conf) in enumerate(zip(bars2, confidences)):
    ax2.text(conf + 0.02, i, f'{conf:.3f}', va='center', fontweight='bold', fontsize=9)

# 3. Scenario Radar: Top 3 Critical Scenarios
ax3 = fig.add_subplot(gs[1, 0], projection='polar')
critical_scenarios = sorted(scenario_names, key=lambda x: scenario_predictions[x]['probability'], reverse=True)[:3]
angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]
for scenario in critical_scenarios:
    values = [scenarios[scenario][f] for f in features]
    # Normalize values for better visualization
    max_vals = [200, 400, 35, 85, 250, 15]
    values_norm = [values[i] / max_vals[i] for i in range(len(values))]
    values_norm += values_norm[:1]
    pred_status = scenario_predictions[scenario]['prediction']
    color = 'red' if pred_status == 'ALARM' else 'green'
    ax3.plot(angles, values_norm, 'o-', linewidth=2, label=f"{scenario} ({pred_status})", color=color, markersize=8)
    ax3.fill(angles, values_norm, alpha=0.15, color=color)
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(features, fontsize=9)
ax3.set_ylim(0, 1)
ax3.set_title('Top 3 Critical Scenarios - Normalized Features', fontsize=12, fontweight='bold', pad=20)
ax3.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
ax3.grid(True)

# 4. Scenario Feature Heatmap
ax4 = axes[1, 1]
scenario_matrix = np.array([[scenarios[s][f] for f in features] for s in scenario_names])
# Normalize for heatmap visualization
scenario_matrix_norm = scenario_matrix / np.array([200, 400, 35, 85, 250, 15])
im = ax4.imshow(scenario_matrix_norm, cmap='RdYlGn_r', aspect='auto')
ax4.set_xticks(np.arange(len(features)))
ax4.set_yticks(np.arange(len(scenario_names)))
ax4.set_xticklabels(features, fontsize=10, fontweight='bold')
ax4.set_yticklabels(scenario_names, fontsize=10)
# Add text annotations
for i in range(len(scenario_names)):
    for j in range(len(features)):
        value = scenario_matrix[i, j]
        text = ax4.text(j, i, f'{value:.0f}', ha="center", va="center", 
                       color="white" if scenario_matrix_norm[i, j] > 0.5 else "black", 
                       fontweight='bold', fontsize=9)
ax4.set_title('Scenario Feature Values (Normalized Color)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax4)
cbar.set_label('Normalized Value', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
plt.show()

# Print scenario predictions summary
print("\n" + "="*80)
print("SCENARIO PREDICTION SUMMARY - EMERGENCY RESPONSE GUIDE")
print("="*80)
for scenario_name in scenario_names:
    pred_info = scenario_predictions[scenario_name]
    print(f"\n📍 {scenario_name}")
    print(f"   Status: {'🚨 ALARM' if pred_info['prediction'] == 'ALARM' else '✅ NO ALARM'}")
    print(f"   Probability: {pred_info['probability']:.4f}")
    print(f"   Confidence: {pred_info['confidence']:.4f}")
    print(f"   Sensor Values: {scenarios[scenario_name]}")
print("\n" + "="*80)